# The `numpy` Module

When dealing with large amounts of data and repetitive  operations, Python lists are not ideal. This is because the contents of a Python list may vary by type and size. This prevents Python from storing the contents in a compact manner or using **vectorised** instructions that can perform operations on a large number of values at once (with the caveat that the values must all have the same type and be laid out in contiguous memory).

NumPy **arrays** solve this problem by enforcing data types and organising data in a dense, continuous block of memory, allowing for faster computations using highly optimised low-level instructions that are supported by the processor.

The following code demonstrates the speedup that can occur using NumPy arrays vs Python lists when performing vector addition :

$$
\begin{bmatrix}
0 \\
1 \\
\vdots \\
9999999
\end{bmatrix}
+
\begin{bmatrix}
0 \\
1 \\
\vdots \\
9999999
\end{bmatrix}
=
\begin{bmatrix}
0 \\
2 \\
\vdots \\
19999998
\end{bmatrix}
$$


<div class="alert">
It is a general convention to import the numpy module in this way

```import numpy as np```

- so that we can differentiate a numpy(vectotised operation) with a Python operation

</div>

In [ ]:
import numpy as np

import time
size = 10_000_000 #1e7

# Python lists.
list1 = list(range(size)) #[0,1,2,3,..9999999]
list2 = list(range(size))

# Timing Python list addition.
start = time.time()
list_result = [list1[i] + list2[i] for i in range(size)]
end = time.time()
list_time = end - start
print(f"Python list addition took: {list_time:.5f} seconds")


# NumPy arrays.
array1 = np.arange(size) #[0 1 2 3 ... 9999999]
array2 = np.arange(size)

# Timing NumPy array addition.
start = time.time()
array_result = array1 + array2
end = time.time()
array_time = end - start
print(f"NumPy array addition took: {array_time:.5f} seconds")
print(f"Numpy perform {list_time/array_time:.5f} times over list")

<div class="alert">

### Analysis of result

- How much performance increase do we get for numpy ?
- Why does numpy array addition perform better than list additions ?
    - Is it `O(1)` vs `O(N)` ?

</div>

In [ ]:
''' 
BlockBox Code
The code below is used for demonstration.
    - repeat previous test with different list/array size
You do not need to know the details
'''

import statistics
import time
import numpy as np
import matplotlib.pyplot as plt

sizes = [10_000_000 * i for i in range(1, 5)]
repeats = 5

list_medians = []
array_medians = []

print(f"{'size':>10} {'list (s)':>12} {'NumPy (s)':>12} {'speedup':>10}")

for size in sizes:
    list1 = list(range(size))
    list2 = list(range(size))
    array1 = np.arange(size)
    array2 = np.arange(size)

    list_times = []
    array_times = []

    for _ in range(repeats):
        start = time.perf_counter()
        # list_result = [a + b for a, b in zip(list1, list2)]
        list_result = [list1[i] + list2[i] for i in range(size)]

        list_times.append(time.perf_counter() - start)
        start = time.perf_counter()
        array_result = array1 + array2
        array_times.append(time.perf_counter() - start)

    list_time = statistics.median(list_times)
    array_time = statistics.median(array_times)
    list_medians.append(list_time)
    array_medians.append(array_time)
    print(f"{size:>10} {list_time:>12.5f} {array_time:>12.5f} {list_time / array_time:>9.1f}x")

In [ ]:
''' 
BlackBox code
The code below is used for demonstration.
    - plot the previous results
You do not need to know this
'''

N = np.array(sizes)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

for ax, times, label, color in [
    (ax1, list_medians, "Python list", "tab:orange"),
    (ax2, array_medians, "NumPy ndarray", "tab:blue"),
]:
    t = np.array(times)
    # Least-squares fit t ≈ a*N + b: a is the per-element coefficient, b the constant term
    a, b = np.polyfit(N, t, 1)

    ax.plot(N, t, "o-", color=color, label="measured (median of 5)")
    ax.plot(N, a * N + b, "--", color="gray", label=f"fit: {a:.2e}*N + {b:.2e}")
    ax.set_title(f"{label}: O(N)")
    ax.set_xlabel("N (number of elements)")
    ax.set_ylabel("time (s)")
    ax.ticklabel_format(style="sci", axis="both", scilimits=(0, 0))
    ax.grid(True, alpha=0.3)
    ax.legend()

plt.tight_layout()
plt.show()

---

## <span style="color:red"> Differences Between Lists and Arrays </span>

#### 1. Homogeneous data type

As mentioned, NumPy arrays enforce data types so all elements have the same data type. This is called the array's **dtype**. Although NumPy supports a wide range of dtypes, for simplicity we shall only consider dtypes that are automatically used to represent Python's built-in `int`, `float` and `str` types.

#### 2. Native multi-dimensionality support

Python's lists are always 1-dimensional and their indexing operator accepts only one index (e.g., `my_list[1]`). To store multi-dimensional data in plain Python, we typically use nested lists, or lists of lists.
For example in  `my_list_of_lists[1][2]`:
-  `my_list_of_lists` is a 1D list which contain lists (each element is a list)
- the first index references one of the nested list and the second index reference the index in the nested list.

However, NumPy arrays have native support for multi-dimensional data and their indexing operator can accept multiple comma-separated indices (e.g., `my_array[1, 2]`).

For 2-dimensional data, the first index is typically called the **row** (axis 0 in numpy) and the second index is called the **column** (axis 1). This corresponds to how the contents of a 2-dimensional array are displayed by default.

The size of an array can be expressed as a tuple of its length in each dimension. This is called the array's **shape**. For example, an array with 2 rows and 3 columns would have a shape of (2, 3).

#### 3. Well-defined uniform shape

A 2D NumPy array must have the same number of columns in every row, so its shape is well-defined; a Python list can have rows of different lengths. For example:
```
Valid numpy array with shape (2,3)
[
    [ 1 2 3 ],
    [ 4 5 6 ]
]

Invalid, shape is not well-defined to form numpy array but valid as a nested list in Python
[
    [1,2,3],
    [4,5]
]
```

Another way to visualise NumPy arrays, as opposed to Python lists, is as finite, geometrically structured containers:

| array dimension | object | 
| -- | -- | 
| 0D | point | 
| 1D | line segment |
| 2D | rectangle |
| 3D | cuboid |
| 4D | hyper-rectangle |
| nD | n-orthotope |

A numpy array has the following metadata:
- shape
- dimension
- dtype
- size

In [33]:
## helper to print array meta data (attributes)
## most the code in this notebook uses this to print a numpy array

import numpy as np
def array_print(array:np.ndarray):
    print(f"{array}")
    print(f"{'dtype':<8}{'shape':<10}{'dim':<5}{'size':<10}")
    print(f"{str(array.dtype):<8}{str(array.shape):<10}{array.ndim:<5}{array.size}")
    print(f"-"*28)

In [ ]:
import numpy as np
a = np.array(
    [
     [1, 2, 3], 
     [4, 5, 6]
    ]
)

array_print(a)
print('value at row 1, col 2:', a[1,2]) # index operator now requires a tuple


---

## <span style="color:red"> Creating Arrays </span>

Here are some ways to create NumPy arrays:

|Category|Functions|
|-|-|
|1. Creating Arrays from Shapes with initial values|`np.empty()`, `np.ones()`, `np.zeros()`|
|2. Creating Arrays from Existing Data|`np.array()`, `np.copy()`, `np.loadtxt()`|
|3. Creating Arrays from a Range|`np.arange()`, `np.linspace()`|
|4. Creating Arrays with random data values|`np.random.randint()`, `np.random.uniform()`|

### - Creating Arrays from Shapes with initialsed values

<div class="alert">
<u>Shape:</u>

A tuple of integers indicating the size of the array along each of its dimensions. For example, a shape of `(3, 2)` represents an array with 3 rows and 2 columns.
</div>

```python
shape(4,) # 1 dimension array
[1,1,1,1] 

shape(2,2) # 2 dimension array of 2 rows, 2 columns
[
    [1,1], 
    [1,1]
]

shape(2,2,2) # 3 dimension array: 2 layer of 2 rows 2 columns
[
    [
        [1, 1],
        [1, 1]
    ],

    [
        [1, 1],
        [1, 1]
    ]
]
shape() # zero dimension , this is a scalar value
1
```

All these functions accept an optional `dtype` keyword argument to set the dtype.

These functions are demonstrated in the code snippets below.

In [ ]:
# np.empty() creates an array of given shape with undefined values
# Important: this means that it may contain random values!
import numpy as np
empty_array = np.empty((2, 3))

empty_array[-1,-1] = 99 # stored as a float
array_print(empty_array)
print()

empty_array = empty_array.astype(str) # type casting an array, changing the data types
array_print(empty_array)
print()



In [ ]:
# np.ones() and np.zeros() will create an array of the given 
# shape filled with 1s and 0s respectively.

array_print(np.ones((5,)))
print()
array_print(np.zeros((5, 1)))

In [ ]:
## array of random numbers 
import numpy as np

random_int=np.random.randint(1,10,(2,2))
array_print(random_int)

random_float = np.random.uniform(1,10,(2,2))
array_print(random_float)

In [ ]:
np.random.rand(2,2) * 10.0

<div class="alert">

#### Question 1

Predict the output of:

1.  `print(np.ones((2,), dtype=str))`
2.  `print(np.ones((1, 1, 1)))`
3.  `print(np.zeros(()))`
4.  `print(np.ones((1,0)))`

</div>

#### Answers
1.
2.
3.
4.

### - Creating Arrays from Existing Data

In [ ]:
# np.array() converts lists, tuples, or other array-like 
# objects into an array. The dtype is set automatically so 
# all elements have the same data type. Type conversions, 
# if needed, are also done automatically.
import numpy as np

# Python list to ndarray
array_print(np.array( 
    [[1, 2, 3], 
     [4, 5, 6]])
)

print()
array_print(np.array(list('ABCDE')))

print()
array_print(np.array(['1965', '2025'], dtype=int))

# tuples to ndarray
print()
array_print(np.array( 
    ((1, 2, 3), 
     (4, 5, 6.0)))
)

print()
array_print(np.array("ABCDE")) #python str to ndarray

print()
array_print(np.array(123)) #python int to ndarray


<div class="alert">

#### 0-D numpy array or Scalar

Note that `np.array("ABCDE")`, `np.array(123)`
- shape ()
- dimenion 0
</div>

<div class="alert">

#### Question 2

Explain why `np.array([[1, 2, 3], [4, 5]])` gives an error.

</div>

### - Copy from an existing array

In [ ]:
# np.copy() creates a deep copy of an existing array.
# Note that arrays are mutable.

a = np.array([[1, 2, 3], 
              [4, 5, 6]])
b = a.copy()

a[1, 2] = 2025
array_print(a)
print()
array_print(b)

<div class="alert">
(Note how `b` is unaffected by changing `a[1, 2]`.)
</div>

### - Reading from a file

In [ ]:
import numpy as np
a = np.loadtxt("unlabelled.csv",delimiter=",")
array_print(a)


### - Creating Arrays of data with a range of values

- using `np.arange(start, end, step)` , similiar to python `range` except 
    - `start,end,step` can be int or float
    - returns an 1D array instead of an iterator

In [ ]:
# np.arange() works like range() but returns an 1D array.
import numpy as np
array_print(np.arange(5))
print()
array_print(np.arange(1.2, 5.5,))
print()
array_print(np.arange(1, 5, 0.2))
print()
array_print(np.arange(10, 1, -0.5))

- using `numpy.linspace( start, stop, num=50, retstep=False)`
    - generates `num` evenly spaced numbers over an interval (includes start and end). 
    - `start` and `stop` can be arrays with the same shape
    - the step size is computed using the given `start`,`stop` and `num`
    - `retstep=True` will return the step size as well as the new array as a tuple `(new_array, step_size)`
    - returns an array that is 1D higher than the shape of the `start`/`end` array as a container

In [ ]:
## generate a 1D array

## start, end are scalar values
array_print(np.linspace(0, 1, 5))## step is (1 - 0)/(5-1), start/end is 0D, 
print()
array_print(np.linspace(60.5, 120.8, 6,)) # step is (120.8-60.5)/(6-1)
print()


In [ ]:
# Generate a 2D array

## start , stop are arrays with the same shape
start = [1, 2] # start row
end = [7, 9] # end row
new_array, step = np.linspace(start, end, 5, retstep=True) 

array_print(new_array)
print(f"Each simensions has its own step size")
print(f"{''.join(f'{s:5.2f}'for s in step)}",sep="\n")

In [ ]:
# Geerate a 3D array

start = np.array([ [0,1], [2,3]]) # 2x2 grid/table , layer 0
end =  np.array([ [10,11], [12,13]]) # 2x2 grid/table, layer 4
new_array, step_size = np.linspace( start, end, 5,retstep=True) # generate 5x (2x2) grids/tables
array_print(new_array)
print()
print("step "+" ".join(f"{value:5.1f}" for value in step_size.reshape(-1,)))

### - Creating array of random numbers

In [ ]:
import numpy as np
''' 
generate an array of random integers with range [start, end)
randint( start, end, shape)
'''
array_print(
    np.random.randint( 1,3, (2,3))
)

''' 
generate an array of random floats with range [start, end)
randint( start, end, shape):
'''
array_print(
    np.random.uniform(5, 10, (2,3))
)

<div class="alert">

#### Question 3

Use a single `np.linspace()` call to generate an array containing:
```
[[[ 1.  2.]
  [ 3.  4.]]

 [[ 2.  4.]
  [ 6.  8.]]

 [[ 3.  6.]
  [ 9. 12.]]]
```

</div>

In [ ]:
# Code here
np.linspace(

)

---
## <span style="color:red"> Accessing Arrays </span>

### - Indexing and Slicing

The indexing `[]` and slicing`[start:end]` operators work for arrays just like they do with Python lists.

- indexing returns an element of the array or list
- slicing returns  an array or a list
- For multi-dimension array, the index and slicing operator use a 'tuple'(comma seperated) instead of an int
```R      
         axis1
        <----->
    ^ [
axis|   [1 2 3]    indexing operator [ axis0, axis1 ]
 0  |   [4 5 6]    slicing operator  [axis0_start:axis0_end, axis1_start:axis1_end]
    v ]

```
    - `array[1,2]` returns 6
    - `array[1:,1:]` returns [5 6]


- However, an important difference is that slicing an array returns a **view** of the array, not a copy. This means that any changes to the slice will affect the original array. To avoid this, it is necessary to use `np.copy()`.

The following demonstrates how changing the contents of view `b` modifies the original array `a`:

In [ ]:
# Original array
import numpy as np
a = np.array([[1, 2, 3], 
              [4, 5, 6]])

print('A:Before:')
array_print(a)

# View of same data
b = a[1:, 1:] 

print("B = A[1:,1:]:")
array_print(b)
print(f" Shared memory is used :{np.shares_memory(a,b)}") ## seperate copies



# Modify view
b[0, 0] = 2025
print()

print('A:After:')
print(a)

<div class="alert">
Note how the value of `a[1, 1]` changed from 5 to 2025 by changing `b[0, 0]`.
</div>

<div class="alert">

#### Question 4

Predict the output of the following program:
```
a = np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9]])
upper_left = a[:-1, :-1]
upper_left[0][0] = 1965
lower_right = a[1:, 1:].copy()
lower_right[1][1] = 2025
print(a)
```

</div>

### - Use an Array(or list) of indices for an axis to extract from the source Array (fancy indexing)
Another powerful feature is that values in specific positions can be extracted by providing array-like values to the indexing operator for each axis. This will return a new array NOT a view.

For example, in the array below: 
```
M = np.array(
[[1 2 3]
 [4 5 6]
 [7 8 9]]
)
 ```
 
 a list or array of indices `[2,0]` can be used  as  an indexing operator on axis 0 in `M[[2, 0],:]`, then a new array containing only the rows at the specified indices are returned as ```[ [7 8 9], [1 2 3]] ```. This is called **fancy indexing**.

i.e. instead of a index or a slice, a list of indices is provided. The numpy array will have a shape based on the array(list) of indexes provided.



In [ ]:
# # import numpy as np
M = np.array([[1, 2, 3], 
              [4, 5, 6], 
              [7, 8, 9]])



# Extract only rows 2 and 0 (i.e., filter out row 1)
_M = M[[2, 0],:] # extract rows 2 and 0 for axis 0 (rows) and all columns(axis 1)
array_print(
    _M     
) # new array shape is (2,3) 

print(np.shares_memory(M,_M)) ## check for same or seperate copies

# Extract only columns 0 and 2 (i.e., filter out column 1)
# print()
# array_print(M[:, [0, 2]]) # extract all the rows (axis 0) and colums 0 and 2, new shape is (3,2)

# Extract (row 2,col 1),  (row 1, col1)
# print()
# array_print( M[ [2,1],[1,2]] ) # return new array with (2,)

# Get three copies of row 1.
# print()
# array_print(M[[1, 1, 1],:]) # shape is (3,3)

<div class="alert">

#### Question 4a: 
Extract the diagonals values of any given square array into a 1D array using fancy indexing

Example in  the square array
```
[[1 2 3]
 [4 5 6]
 [7 8 9]]
 ```
 the diagonals array is `[1 5 9]`
</div>

Complete the code block below:

In [ ]:
import numpy as np, random
n = random.randint(1,10)
sq_array = np.random.randint( [1]*n,[10]*n, (n,n))
print(sq_array)

## extract and print the diagonals of sq_array
sq_array[ 
    ## complete the code
]



### - Using Boolean Masking

If the index is a list or array of bool values that match the array's size or shape, then a new array containing only values where the corresponding index is `True` is returned. This is called **Boolean masking**. This is typically used to filter an array based on a condition.

In [ ]:
import numpy as np
a = np.array([[1, 2, 3], 
              [4, 5, 6], 
              [7, 8, 9]])

# Extract only rows 0 and 2 (i.e., filter out row 1)
print(a[[True, False, True],:])

# Extract only columns 0 and 2 (i.e., filter out column 1)
print()
print(a[:, [True, False, True]])



In [ ]:
## Extract only even numbers from my_array
import numpy as np

my_array = np.array(
    [   [1, 2, 3], 
        [4, 5, 6], 
        [7, 8, 9]
    ])

# step 1: perform %2 operation on every element in my_array

mod_mask = my_array % 2 
print(mod_mask) # mod_mask  has the same shape as my_array every element is now the remainder after diving by 2

#step 2: broadcast  equality operation ( == 0) on mod_mask, even_mask is also same shape as my_array, every element in even_mask is now a boolean value
even_mask = mod_mask == 0
print(even_mask)

#step 3: use the resulting result as a boolean mask to extract from my_array
print( my_array[even_mask])

<div class="alert">

#### Question 4b: 
Now, use a single line to extract even numbers from `my_array`
</div>

In [ ]:
# code here
my_array[ 
    # complete the code here
]

### Summary of numpy array access

In 
```
my_array = np.array(
    [   [1, 2, 3], 
        [4, 5, 6], 
        [7, 8, 9]
    ])
```
the numpy index operator `[]` supports:
- indexes: `my_array[1,2]` $\rightarrow$ `6`
- slices:  `my_array[1:,:-1]` $\rightarrow$ `[[4,5],[7,8]]`
- indices array/lists (fancy indesing): `my_array[[2,0],[1,2]]` $\rightarrow$ `[8, 3]`
    - picks elements at (2,1) and (0,2)
- boolean masking: `my_array[ my_array % 2 == 0]` $\rightarrow$ `[2, 4, 6, 8]`

<div class='alert'>

#### Question 5

1. Generate a array of shape `(13,3`)` of random integers from 1 to 99

2. Extract only the odd numbers from the even rows in the array


</div>

In [ ]:
## Code here

## <span style="color:red"> Manipulating Arrays </span>

Here are some ways to manipulate NumPy arrays:

|Function|Description|
|-|-|
|`np.append()`|Adds elements to copy of array.|
|`np.concatenate()`|Joins arrays along an axis.|
|`np.delete()`|Deletes elements.|
|`np.insert()`|Inserts values at specified positions.|
|`np.reshape()`|Changes shape of array without changing data.|
|`np.resize()`|Changes size of array and fills with repeated data if needed.|
|`np.transpose()`|Switches the axes of an array. For 2D, switching the rows and columns |
|`np.sort()`|Sorting (non-inplace)|

<div class="alert">

Note that unlike Python lists, array shape needs to be well defined after you mutate it. 

For example adding/removing an integer to/from  any row or column in `my_array` will deform the array's shape

```
my_array = np.array(
    [   [1, 2, 3], 
        [4, 5, 6], 
        [7, 8, 9]
    ])
```
Therefore, `append,delete,insert` by default will flatten (re-shape to a 1D array) before you can add and remove an element from it. You can then re-shape (discuss later) the new array into a well defined shape.

</div>

### `np.append()`
<div class="alert">

<u>**NOTE**:</u>

unlike the append used in a Python list, numpy append  is actually a wrapper for concatenate. It does not perform an inplace append to the original array. It creates a copy of the original array and **concatenates** a new array to it. 


`np.append(A, B, axis=None)` $\equiv$ `np.concatenate( (A,B), axis=None)` 

- default `axis = None` means flattern to a 1D array
- `axis = 0` , append/concatenate rows, number of columns must be the same
- `axis = 1`, append/concatenate columns, number of rows must be the same


</div>

In [ ]:
# np.append() joins elements to copy of array.
# If the optional axis argument is None or not specified, the 
# array is flattened. Otherwise, the elements are joined to the array 
# along the specified axis.
import numpy as np

a = np.array([[1, 2, 3], 
              [4, 5, 6], 
              [7, 8, 9]])

print("A:")
array_print(a)

# Example: append an element to a flattened array
# m = np.append(a,1940) # default axis=None
# array_print(m)
# # append a list to a flattened array
# m = np.append(a, [1965, 2017, 2025])
# array_print(m)

# Example: append row 
# note that a's dimension is 2, the array to be appended must also be dimension 2.
# a's shape is (3,3), you can append any (n,3) shapes array to it's axis 0 (rows), 
# append a (3,) array to a will fail

m = np.append(a, [[1965, 2017, 2025]], axis=0) # cols(axis 1) must match a
print()
array_print(m)

# Example: append column
# note that a's dimension is 2, the array to be appended must also be dimension 2.
# a's shape is (3,3), you can append any (3,n) shapes array to a's axis 1 (columns), 
# append (3,) will fail

# m = np.append(a, [[1965], [2017], [2025]], axis=1) # rows(axis 0) must match a
# print()
# print(m)

### `np.concatenate( (A,B) )`
- default `axis=0`, i.e. join rows

In [ ]:
# np.concatenate() joins arrays along an axis (default 0).
# If the optional axis argument is None, the arrays are 
# flattened. Otherwise, the arrays are joined along the 
# specified axis.

# Example: join flattened arrays
import numpy as np

m1 = np.array([[1, 2], [3, 4], [5, 6]]) #(3,2)
m2 = np.array([1965, 2017, 2025]) #(1,3)
m = np.concatenate((m1, m2), axis=None)
array_print(m)


In [ ]:
# Example: join rows
# a shape is (3,3)
import numpy as np
m1 = np.array([[1, 2, 3], 
              [4, 5, 6], 
              [7, 8, 9]]) #(3,3)
m2 = np.array([[11, 12, 13], [14, 15, 16]]) #(2,3) cols match
m = np.concatenate((m1, m2) )# defaul axis is 0
print()
array_print(m)

In [ ]:
# Example: join columns
import numpy as np
m1 = np.array([[1, 2, 3], 
               [4, 5, 6], 
               [7, 8, 9]]) #(3,3)

m2 = np.array([[11, 12], [13, 14], [15, 16]]) #(3,2) row match, add  columns to each row
m = np.concatenate((m2, m1), axis=1)
print()
array_print(m)

In [ ]:
a = np.array( [[1,2,3], [1,2,3]]) # (2,3)
b = np.array([[4,5,6],[-1,-1,-1]]) #(2,3)
c = np.array([(7,8,9),[-1,-1,-1]]) #(2,3)
d = np.array([(99,98,97),[-1,-1,-1]]) #(2,3)
m1 = np.concatenate( (a,b,c,d), axis = 0) # join rows
print("m1:")
array_print(m1)
print()
m2 = np.concatenate( (a,b,c,d), axis = 1) # join cols
print("m2:")
array_print(m2)

<div class="alert">

#### Question 6

In the following two calls, `axis` is an `int` and is not `None`:

1. `np.append(m1, m2, axis)`
2. `np.concatenate((m1, m2), axis)`

State the condition that must be satisfied by `m1.shape` and `m2.shape` so the calls are valid.

</div>

> Hint
>
>if axis = 0 (join rows)
>    - m1 and m2 must have ???

>if axis = 1(join colums)
>    - m1 and m2 must have ???

### `np.delete()`
- default, `axis=None` 

In [ ]:
# # np.delete() deletes elements with given indices along 
# # given axis. If the optional axis argument is None or not 
# # specified, the array is flattened.
# import numpy as np
# a = np.array([[1, 2, 3], 
#               [4, 5, 6], 
#               [7, 8, 9]])
array_print(a)

# # delete at index 4 after flatening
# array_print(np.delete(a,4))

# # Delete row 1
# array_print(np.delete(a, 1, axis=0))

# # Delete column 1
# print()
# array_print(np.delete(a, 1, axis=1))

# # # Delete columns 1 and 2 (using fancy indexing)
# print()
# array_print(np.delete(a, [1, 2], axis=1))

# # # Delete rows 0 and 2 (using Boolean masking)
# # print()
# # array_print(np.delete(a, [True, False, True], axis=0))


### `np.insert()`
- default `axis=None`

In [ ]:
# # np.insert() inserts values before given indices along 
# # given axis. If the optional axis argument is None or not 
# # specified, the array is flattened.

# a = np.array([[1, 2, 3], 
#               [4, 5, 6], 
#               [7, 8, 9]])
# array_print(a)

# # # Insert before item 5 in flattened array
# # print()
# # array_print(np.insert(a, 5, 2025, axis=None))

# # Inserts row between rows 0 and 1
# # array_print(np.insert(a, 1, [1965, 2017, 2025], axis=0)) # cols must match a

# # Inserts column between columns 0 and 1
# # print()
# # array_print(np.insert(a, 1, [1965, 2017, 2025], axis=1)) # rows must match a

# # # Inserts before columns 1 and 2 (using fancy indexing)
# # print()
# array_print(np.insert(a, [1, 2], [[11, 12], [13, 14], [15, 16]], 
#                 axis=1)) # rows must match



### `np.reshape(), np.resize()`

In [ ]:
# np.reshape() and np.resize() both resize arrays.
# np.reshape() requires the new shape to have exactly the 
# same number of values. 
# np.resize() lets the new shape be 
# larger or smaller. If the new shape is larger, extra values 
# are filled with copies of the array.

m = np.array([[1, 2], [3, 4], [5, 6], [7, 8]])
array_print(m)

# Reshape to multi-dimensional array.
# array_print(np.reshape(m, (2, 2, 2)))

# Reshape to multi-dimensional array. (Same as above.)
# print()
# array_print(np.resize(m, (2, 2, 1)))

# # Reshape to larger array with repeated copies of array.
print()
array_print(np.resize(m, (2, 6)))

### `np.transpose()`
- default `axes=None`, means the axes are reversed,
- for each element in a
    - add to the transposed index of the new array, i.e row becomes cols, cols becomes row
- Example:
```
a = np.array(
    [
        [10,9,8],
        [7,6,5]
    ]
)
new_array = np.empty( a.shape[::-1]) # create an empty new array with the transposed shape
for i in range(shape[0]): # axis 0
    for j in range(shape(1)): #axis 1
        old_ind = (i,j)
        new_ind = old_ind[::-1] # reverse the axes
        new_array[new_ind] = a[old_ind]
```


In [ ]:
# np.transpose() swaps the rows and columns for 2D arrays.
# For multi-dimensional arrays, the axes are reversed unless 
# axes argument is provided. 

# Straightforward 2D transpose.
# a = np.array([[1, 2, 3], 
#               [4, 5, 6], 
#               [7, 8, 9]])
# array_print(a)

# print(np.transpose(a))

m = np.array([  [
                    [1, 2, 3], 
                    [4, 5, 6]
                ], 
                [
                    [7, 8, 9],
                    [10, 11, 12]
                ]
              ])

array_print(m)
# # Multi-dimensional transposes.
# print()
array_print(np.transpose(m, axes=None))
print()
# array_print(np.transpose(m, axes=[1, 0, 2]))

### - `np.sort()`
    - default sort by last axis(-1)(col)
        for each row, sort by col
    - axis=None, flatten then sort

In [2]:
import numpy as np
# np.random.seed(42)
A = np.random.randint( 1, 10, (3,4) )
print(A)
# np.sort(A) # for each row sort the cols
np.sort(A,axis=0) # for each col sort the row
# np.sort(A,axis=None)
np.sort(A, axis=None, descending=True) #numpy 2.0 


[[9 9 2 2]
 [6 3 5 3]
 [7 8 5 5]]


array([9, 9, 8, 7, 6, 5, 5, 5, 3, 3, 2, 2], dtype=int32)

<div class="alert">
    
#### Question 7

Array `a` is defined as follows:

```
a = np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9]])
```

Use a single `np.insert()` call to manipulate `a` into an array containing:

```
[[ 1  2  3]
 [ 4  5  6]
 [-1 -1 -1]
 [ 7  8  9]]
```

</div>

In [ ]:
a = np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9]])
np.insert(a, 2,[-1,-1,-1],axis=0)

# Double-click this cell and replace this placeholder with your answer.

<div class="alert">
    
#### Question 8

Use as few calls as possible to generate an array containing:
```
[[0 0 0]
 [1 1 1]]
```
You may not use `np.array()` directly.

</div>

In [ ]:
## Code here


<div class="alert">
    
#### Question 9

Use as few calls as possible to generate an array containing:
```
[[0 3 1 4]
 [1 4 2 0]
 [2 0 3 1]]
```
You may not use `np.array()` directly.

</div>

In [ ]:
# Double-click this cell and replace this placeholder with your answer.
np.transpose(np.resize( np.arange(5),(4,3)))


## <span style="color:red"> Performing Mathematical Operations </div>

NumPy allows for mathematical operations (including addition, subtraction, multiplication and division via the usual operators) to be performed element-wise on a large number of values at once. (We saw this performance gain earlier). 

For this to happen the operands must have the same shape.

In [ ]:
import numpy as np

m1 = np.array([1, 2, 3]) # (3,)
m2 = np.array([4, 5, 6]) # (3,)

print(m1 + m2)
print()
print(m1 - m2)
print()
print(m1 * m2)
print()
print(m1 / m2)
print()
print(m2 % m1)

### Broadcasting

NumPy also supports a feature called **broadcasting**, which automatically expands smaller arrays to be compatible with larger arrays when performing mathematical operations. The simplest application of broadcasting allows us to combine a scalar with an array:

In [ ]:
import numpy as np
m1 = np.array([1, 2, 3]) # shape is (3,) with 1 dimension
m2 = 2 # shape is () with 0 dimension

array_print( m1 * m2)

#### Broadcasting Mechanics

Broadcasting compares the array shapes of the operands **right to left** (starting from trailing dimensions).

1. **Align:** Align shape tuples starting from the rightmost dimension.
2. **Pad:** If dimensions differ in count, prepend `1`s to the left of the shorter shape.
3. **Check & Virtually Expand:**
   * Compare each dimension pair-by-pair from right to left.
   * Dimensions are compatible if they are **equal** or if **one of them is `1`**.
      * If a `1` is paired with $n$ ($n > 1$), elements along that axis are **virtually expanded** to size $n$.
   * If dimensions differ and neither is `1`, NumPy throws a `ValueError`.

```python
# Example: array A has shape (3, 4, 5) and array B has shape (5,)
#                       A           B
# 1. Align & Pad:  (3, 4, 5) vs (1, 1, 5)
# 2. Check:        Axis 2: 5 == 5 (Valid)
#                  Axis 1: 4 vs 1 (Valid -> Virtually expands B's 1 row to 4)
#                  Axis 0: 3 vs 1 (Valid -> Virtually expands B's 1 layer to 3)
```

---

In [ ]:
import numpy as np
A = np.random.randint( 1, 3, (3,4,5) )
B = np.random.randint( 10, 11, (5,) )
array_print(
    A+B
)

| Shapes (a and b)| Promotion |Compatible? | Result shape |
|---|---|:---|---:|
| `(3, 2)` and `(2,)`  |b $\rightarrow$ `(1,2)` | Yes, b row is broadcast | `(3, 2)` |
| `(3, 2)` and `(1, 2)`| | Yes, b row is broadcast | `(3, 2)` |
| `(3, 2)` and `(3, 1)`| | Yes, b column is broadcast | `(3, 2)` |
| `(3, 2)` and `(3,)`  |b $\rightarrow$ `(1,3)` |  No | — |


Broadcasting is a **conceptual** expansion for the operation; NumPy usually doesn’t need to create repeated copies of the smaller operand.

<div class="alert">

#### Question 10

1. Without writing code, show whether the following array operations works ?

    a. 
    ```R 
    [1 2 3] * 2
    ```
    b.
    ```R
    [
        [
            [ 1 2 ]
        ]
        [
            [ 3 4 ]
        ]
        [
            [ 5 6 ]
        ]
    ]
    +
    [
        [
            [1 2]
            [3 4]
        ]
    ]
    ```
2. Predict the output of the following code:

```R
print(np.zeros((1, 2, 1)) + np.ones((1, 1, 3)))
```

</div>

In [ ]:
m1 = np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9]]) # (3,3)
m2 = np.array([[1965, 2017, 2025]]) # (1,3)

# Operands differ only in dimension 0,
# and one of the operands has length 1 in that dimension.
print(m1.shape)
print(m2.shape)

# Broadcasting automatically expands m2 so it is repeated 3 
# times in dimension 0.
print()
print(m1 + m2)

### Element-wise mathematical operations supported by NumPy:

|Category|Functions|
|-|-|
|Rounding values|`np.ceil()`, `np.floor()`, `np.round()`, `np.trunc()`|
|Arithmetic operations|`np.exp()`, `np.log()`, `np.power()`, `np.sqrt()`|


These functions are demonstrated in the code snippets below.

In [ ]:
import numpy as np
a = np.array([-3.72, -2.56, -1.28, 0.0, 1.28, 2.56, 3.72])


# np.ceil() rounds up to nearest integer.
print('Ceiling:', np.ceil(a))

# # np.floor() rounds down to nearest integer.()
# print()
# print('Floor:', np.floor(a))

# # np.round() rounds to nearest integer by default.
# print()
# print('Round:', np.round(a))

# # The number of decimal places can also be set.
# print()
# print('Round to 1dp:', np.round(a, 1))

# # np.trunc() rounds toward zero.
# print()
# print('Truncate:', np.trunc(a))

In [ ]:
import numpy as np
a = np.array([2, 3, 4])
b = np.array([4, 5, 6])

# # np.exp() calculates e^x.
# print()
# print('Exponential:', np.exp(a))

# # np.log() calculates the natural logarithm.
# print()
# print('Natural logarithm:', np.log(a))


# # np.sqrt() calculates the square root.
# print()
# print('Square root:', np.sqrt(a))

# # np.power() raises elements in an array to powers
# # given in a second array, element-wise.
print()
print('Power (element-wise):', np.power(a, b))


# # The elements can also be raised to a fixed power
# # through broadcasting.
# print()
# print('Power (squared):', np.power(a, 2))

### Reduction mathematical operations supported by NumPy:
- perform on the whole array or on an axis(dimension)

|Category|Functions|
|-|-|
|Summing values|`np.sum()`, `np.cumsum()`|
|Finding maxima/minima|`np.amax()`, `np.amin()`|
|Basic statistics|`np.mean()`, `np.std()`|
|Linear Algebra|`np.dot()`, `np.matmult()`|



#### Sum Operation
- operates on the whole array or on an axis
- default `axis=None`, operates on the whole array returns a scalar value
- if a axis is specified, returns a 1D array
    

In [ ]:
a = np.array([[1, 2, 3],
              [4, 5, 6],
              [7, 8, 9]])
array_print(a)

# np.sum() adds all elements by default. 
print('Sum of all elements:', np.sum(a)) # 

# # The sum can also be performed along a given axis 
# # using the axis parameter.
# print()
# print('Sum along axis 0:', np.sum(a, axis=0)) # collapse the rows
# print('Sum along axis 1:', np.sum(a, axis=1)) # collapse the columns

# # np.cumsum() flattens the array and calculates the
# # running sum by default.
# print()
# print('Cumulative sum of flattened array:', 
#       np.cumsum(a))

# # The cumulative sum can also be performed along a
# # given axis using the axis parameter.
# print()
# print('Cumulative sum along axis 0:')
# print(np.cumsum(a, axis=0))

#### Statistical Operations
- operates on the whole array or on an axis
- default `axis=None`, operates on the whole array returns a scalar value
- if a axis is specified, returns a 1D array
- `np.amax()` $\equiv$ `np.max()` 
- `np.amin()` $\equiv$ `np.min()` 


In [ ]:
a = np.array([[1, 2, 3],
              [5, 6, 4],
              [9, 7, 8]])

# np.mean() calculates the average of the entire array
# by default.
print('Mean of entire array:', np.mean(a))

# The mean can also be calculated along a given axis 
# using the axis parameter.
print()
print('Mean along axis 0:', np.mean(a, axis=0))
print('Mean along axis 1:', np.mean(a, axis=1))

# np.std() calculates the standard deviation of the 
# entire array by default.
print('Standard deviation of entire array:', np.std(a))

# Standard deviation along rows/columns
print('Standard deviation along rows:')
print(np.std(a, axis=0))  # [3. 3. 3.]

# Combined statistics example

data = np.array([15, 23, 42, 35, 28, 19, 33])
print('Summary statistics:')
print(f'Mean: {np.mean(data):.2f}')
print(f'Std Dev: {np.std(data):.2f}')
print(f'Min: {np.amin(data)}')
print(f'Max: {np.amax(data)}')
print(f'Median: {np.median(data)}')

In [ ]:
a = np.array([[1, 2, 3],
              [4, 5, 6],
              [7, 8, 9]])

# np.amax() finds the maximum value of the entire array
# by default.
print('Maximum of entire array:', np.amax(a))

# The maximum can also be calculated along a given axis 
# using the axis parameter.
print()
print('Maximum along axis 0:', np.amax(a, axis=0))
print('Maximum along axis 1:', np.amax(a, axis=1))

# np.amin() finds the minimum value of the entire array
# by default.
print()
print('Minimum of entire array:', np.amin(a))

# The minimum can also be calculated along a given axis 
# using the axis parameter.
print()
print('Minimum along axis 0:', np.amin(a, axis=0))
print('Minimum along axis 1:', np.amin(a, axis=1))

#### Dot product of two arrays, A and B
- if A and B are 1D and same size, then it's the sum of products, `np.sum(A*B)`
- if A and B are 2D and A's shape[1] == B's shape[0], then its a matrix multiplication `A@B`
    - `np.matmul(A,B)` $\equiv$ `A@B`

In [ ]:
import numpy as np
a = np.array([2, 3, 4]) #(3,)
b = np.array([4, 5, 6])


# np.dot() performs a dot product.
print('Dot product:', np.dot(a,b))
print("Sum of products",np.sum(a*b))

# Matrix multiplication 
# A = np.resize(a, (2,2))
# B = np.resize(b,(2,5))
array_print(np.dot(A,B))
print(A@B)
print(np.matmul(A,B))


## Consolidation Exercises

<div class="alert ">
    
### Question 11
	
The daily sales of a store are stored in an array named `daily_sales`. Calculate the store’s cumulative sales after each day.

</div>

In [ ]:
# Code 


<div class="alert">
    
### Question 12
The ratings for 10 restaurants are rated based on the number of stars given by customers in a survey

a) Generate random integers from 1 to 5 inclusive of 100 customer ratings for each of the 10 restaurants and store them in an array `ratings`

b) Generat the average ratings for each of the the restaurants and store them in an array `mean_ratings`

c) Round off the star ratings to the nearest half star ( i.e 3.3 -> 3.5, 3.2-> 3.0)

</div>



In [ ]:
# Code

<div class="alert">
    
### Question 13

The Cartesian coordinates $(x, y)$ of 18 points on a Euclidean plane are stored in a NumPy array of shape (18, 2).

a) Generate random float coordinates in the range from 1 up to 20 for all 18 ponts and store them in a numpy array named `data_points`

b) The centroid of a set of points is the point whose $x$-coordinate is the arithmetic mean of the $x$-coordinates of all points, and whose $y$-coordinate is the arithmetic mean of the $y$-coordinates of all points.

For the 18 data points, the centroid $(\bar{x}, \bar{y})$ is defined as:

$$ \bar{x}=\frac{1}{18}\sum_{i=1}^{18}x_i \qquad \bar{y}=\frac{1}{18}\sum_{i=1}^{18}y_i $$

Calculate the centroid of these 18 points and store it in a variable named `centroid`.

</div>

In [ ]:
## Code 


<div class="alert">

The Euclidean distance between two points $P(x_1,y_1)$ and $Q(x_2,y_2)$ on a Euclidean plane is the length of the straight-line segment joining them, calculated using the formula:

$$
d(P,Q)=\sqrt{(x_2-x_1)^2+(y_2-y_1)^2}
$$
    
c)	Using only `numpy` functions, calculate the distance of each data point in `data_points` from `centroid` and store the results in an array named `distances`. You cannot use any explicit loop construct.

</div>

In [ ]:
#Code 

<div class="alert">
    
c)	Find the data point that is closest to the centroid.

</div>

In [ ]:
# Code